# Regression in PySpark's MLlib Project

Now it's time to put what you've learned to into action with a REAL project! 

You have been hired as a consultant to a cement production company who wants to be able to improve their customer experience around a number of areas like being able to provide recommendations to cusomters on optimal amounts of certian ingredients in the cement making process and perhaps even create an application where users can input their own values and received a predicted cement strength!

I have provided a list of question below to help guide you through this project but feel free to deviate and make this project your own! But first, a bit about this dataset.

### About this dataset 
This dataset contains 1030 instances of concrete samples, containing 9 attributes (8 continuous and 1 discreate), and 1 continuous quantitative output variable. There are no missing attribute values.

I've also provided the variable name, variable type, the measurement unit and a brief description of each variable in the dataset. The concrete compressive strength is the outcome variable for our analysis. The order of this listing corresponds to the order of numerals along the rows of the database.

Name -- Data Type -- Measurement -- Description

- Cement -- quantitative -- kg in a m3 mixture -- Input Variable 
- Blast Furnace Slag -- quantitative -- kg in a m3 mixture -- Input Variable 
- Fly Ash -- quantitative -- kg in a m3 mixture -- Input Variable 
- Water -- quantitative -- kg in a m3 mixture -- Input Variable 
- Superplasticizer -- quantitative -- kg in a m3 mixture -- Input Variable 
- Coarse Aggregate -- quantitative -- kg in a m3 mixture -- Input Variable 
- Fine Aggregate -- quantitative -- kg in a m3 mixture -- Input Variable 
- Age -- quantitative -- Day (1~365) -- Input Variable 
- Concrete compressive strength -- quantitative -- MPa -- Output Variable

**Source:** https://www.kaggle.com/maajdl/yeh-concret-data

**Dataset Name:** Concrete_Data.csv

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Cement Strength Prediction").getOrCreate()

cores = spark._jsc.sc().getExecutorMemoryStatus().keySet().size()
print("You are working with", cores, "core(s)")
spark

You are working with 1 core(s)


In [2]:
from pyspark.ml.feature import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.classification import *
from pyspark.ml.evaluation import *
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline 

In [3]:
concrete_data = spark.read.csv('data/Concrete_Data_Yeh.csv', inferSchema=True, header=True)

In [4]:
concrete_data.show(5, truncate=False)

+------+-----+------+-----+----------------+---------------+-------------+---+-----+
|cement|slag |flyash|water|superplasticizer|coarseaggregate|fineaggregate|age|csMPa|
+------+-----+------+-----+----------------+---------------+-------------+---+-----+
|540.0 |0.0  |0.0   |162.0|2.5             |1040.0         |676.0        |28 |79.99|
|540.0 |0.0  |0.0   |162.0|2.5             |1055.0         |676.0        |28 |61.89|
|332.5 |142.5|0.0   |228.0|0.0             |932.0          |594.0        |270|40.27|
|332.5 |142.5|0.0   |228.0|0.0             |932.0          |594.0        |365|41.05|
|198.6 |132.4|0.0   |192.0|0.0             |978.4          |825.5        |360|44.3 |
+------+-----+------+-----+----------------+---------------+-------------+---+-----+
only showing top 5 rows



In [5]:
concrete_data.printSchema()

root
 |-- cement: double (nullable = true)
 |-- slag: double (nullable = true)
 |-- flyash: double (nullable = true)
 |-- water: double (nullable = true)
 |-- superplasticizer: double (nullable = true)
 |-- coarseaggregate: double (nullable = true)
 |-- fineaggregate: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- csMPa: double (nullable = true)



In [6]:
concrete_data.count(), len(concrete_data.columns)

(1030, 9)

In [7]:
concrete_data.select([count(when(isnan(c), c)).alias(c) for c in concrete_data.columns]).limit(5).toPandas()

,cement,slag,flyash,water,superplasticizer,coarseaggregate,fineaggregate,age,csMPa
0,0,0,0,0,0,0,0,0,0


In [8]:
def MLRegressDFPrep(df,input_columns,dependent_var,treat_outliers=True):

    renamed = df.withColumnRenamed(dependent_var,'label')
    
    # Make sure dependent variable is numeric and change if it's not
    if str(renamed.schema['label'].dataType) != 'IntegerType':
        renamed = renamed.withColumn("label", renamed["label"].cast(FloatType()))
    
   # Convert all string type data in the input column list to numeric
    # Otherwise the Algorithm will not be able to process it
    numeric_inputs = []
    string_inputs = []
    for column in input_columns:
        if str(renamed.schema[column].dataType) == 'StringType':
            new_col_name = column+"_num"
            string_inputs.append(new_col_name)
        else:
            numeric_inputs.append(column)
            indexed = renamed
            
    if len(string_inputs) != 0: # If the datafraem contains string types
        for column in input_columns:
            if str(renamed.schema[column].dataType) == 'StringType':
                indexer = StringIndexer(inputCol=column, outputCol=column+"_num") 
                indexed = indexer.fit(renamed).transform(renamed)
    else:
        indexed = renamed
        
            
    if treat_outliers == True:
        print("We are correcting for non normality now!")
        # empty dictionary d
        d = {}
        # Create a dictionary of quantiles
        for col in numeric_inputs: 
            d[col] = indexed.approxQuantile(col,[0.01,0.99],0.25) #if you want to make it go faster increase the last number
        #Now fill in the values
        for col in numeric_inputs:
            skew = indexed.agg(skewness(indexed[col])).collect() #check for skewness
            skew = skew[0][0]
            # This function will floor, cap and then log+1 (just in case there are 0 values)
            if skew > 1:
                indexed = indexed.withColumn(col, \
                log(when(df[col] < d[col][0],d[col][0])\
                .when(indexed[col] > d[col][1], d[col][1])\
                .otherwise(indexed[col] ) +1).alias(col))
                print(col+" has been treated for positive (right) skewness. (skew =)",skew,")")
            elif skew < -1:
                indexed = indexed.withColumn(col, \
                exp(when(df[col] < d[col][0],d[col][0])\
                .when(indexed[col] > d[col][1], d[col][1])\
                .otherwise(indexed[col] )).alias(col))
                print(col+" has been treated for negative (left) skewness. (skew =",skew,")")
                
    # Vectorize your features
    features_list = numeric_inputs + string_inputs
    assembler = VectorAssembler(inputCols=features_list,outputCol='features')
    final_data = assembler.transform(indexed).select('features','label')
        
    return final_data

In [9]:
input_columns = concrete_data.columns[:-1]
dependent_var = concrete_data.columns[-1]

final_data = MLRegressDFPrep(concrete_data, input_columns, dependent_var)
final_data.show(5, truncate=False)

We are correcting for non normality now!


age has been treated for positive (right) skewness. (skew =) 3.2644145354168086 )


+----------------------------------------------------------+-----+
|features                                                  |label|
+----------------------------------------------------------+-----+
|[540.0,0.0,0.0,162.0,2.5,1040.0,676.0,3.367295829986474]  |79.99|
|[540.0,0.0,0.0,162.0,2.5,1055.0,676.0,3.367295829986474]  |61.89|
|[332.5,142.5,0.0,228.0,0.0,932.0,594.0,5.602118820879701] |40.27|
|[332.5,142.5,0.0,228.0,0.0,932.0,594.0,5.902633333401366] |41.05|
|[198.6,132.4,0.0,192.0,0.0,978.4,825.5,5.8888779583328805]|44.3 |
+----------------------------------------------------------+-----+
only showing top 5 rows



## 1. Which features are the strongest predictors of cement strength?

Build your own ML model to figure this one out! This would be good information to give to our client so the sales reps can focus their efforts on certian ingredients to provide recommendations on. For example, if our clients had a customer that was struggling with their cement breaking, we could trouble shoot with them by starting with the factors that we know are important. 

In [10]:
train, test = final_data.randomSplit([0.7,0.3], seed=42)

In [11]:
from pyspark.ml.regression import *

regressor = LinearRegression()
fitModel = regressor.fit(train)

In [12]:
predictions = fitModel.transform(test)

evaluator = RegressionEvaluator(metricName="rmse")
rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 7.64529


In [13]:
def RegressTrainEval(regressor):

    def FindMtype(regressor):
        # Intstantiate Model
        M = regressor
        # Learn what it is
        Mtype = type(M).__name__
        
        return Mtype
    
    Mtype = FindMtype(regressor)
#     print('\033[1m' + Mtype + ':' + '\033[0m')


    if Mtype == "LinearRegression":
        
        #first without cross val
        fitModel = regressor.fit(train)

        # Load the Summary
        trainingSummary = fitModel.summary
        
        # Print the coefficients and intercept for linear regression
        print('\033[1m' + "Linear Regression Model Training Summary without cross validation:"+ '\033[0m')
        print(" ")
        print("Intercept: %s" % str(fitModel.intercept))
        print("")
        print("Coefficients: ")
        coeff_array = fitModel.coefficients.toArray()
        # Convert from numpy array to list
        coeff_list = []
        for x in coeff_array:
            coeff_list.append(float(x))
        result = spark.createDataFrame(zip(input_columns,coeff_list), schema=['feature','coeff'])
        print(result.orderBy(result["coeff"].desc()).show(truncate=False))



        # Summarize the model over the training set and print out some metrics
        print("numIterations: %d" % trainingSummary.totalIterations)
        print("objectiveHistory: (scaled loss + regularization) at each iteration \n %s" % str(trainingSummary.objectiveHistory))
        print("")
        
        # Print the Errors
        print("Training RMSE: %f" % trainingSummary.rootMeanSquaredError)
        print("Training r2: %f" % trainingSummary.r2)
        print("")
        

        # Now load the test results
        test_results = fitModel.evaluate(test)

        # And print them
        print("Test RMSE: {}".format(test_results.rootMeanSquaredError))
        print("Test r2: {}".format(test_results.r2))
        print("")
        
        #Now train with cross val
        paramGrid = (ParamGridBuilder() \
#              .addGrid(regressor.maxIter, [10, 15]) \
             .addGrid(regressor.regParam, [0.1, 0.01]) \
             .build())
        
        #Evaluator
        revaluator = RegressionEvaluator(metricName="rmse")
        
        #Cross Validator requires all of the following parameters:
        crossval = CrossValidator(estimator=regressor,
                                  estimatorParamMaps=paramGrid,
                                  evaluator=revaluator,
                                  numFolds=5) # 3 is best practice
        
        print('\033[1m' + "Linear Regression Model Summary WITH cross validation:"+ '\033[0m')
        print(" ")
        # Run cross validations
        fitModel = crossval.fit(train)
        
        #save model
        global LR_BestModel 
        LR_BestModel = fitModel.bestModel
        
        print("Coefficients: ")
        coeff_array = LR_BestModel.coefficients.toArray()
        # Convert from numpy array to list
        coeff_list = []
        for x in coeff_array:
            coeff_list.append(float(x))
        result = spark.createDataFrame(zip(input_columns,coeff_list), schema=['feature','coeff'])
        print(result.orderBy(result["coeff"].desc()).show(truncate=False))
        
        # Get Model Summary Statistics
        ModelSummary = fitModel.bestModel.summary
        
        print("Coefficient Standard Errors: ")
        coeff_ste = ModelSummary.coefficientStandardErrors
        result = spark.createDataFrame(zip(input_columns,coeff_ste), schema=['feature','coeff std error'])
        print(result.orderBy(result["coeff std error"].desc()).show(truncate=False))
        print(" ")
        print("P Values: ") 
        # Then zip with input_columns list and create a df
        pvalues = ModelSummary.pValues
        result = spark.createDataFrame(zip(input_columns,pvalues), schema=['feature','P-Value'])
        print(result.orderBy(result["P-Value"].desc()).show(truncate=False))
        print(" ")
        
        # Use test set here so we can measure the accuracy of our model on new data
        ModelPredictions = fitModel.transform(test)
        
        # cvModel uses the best model found from the Cross Validation
        # Evaluate best model
        test_results = revaluator.evaluate(ModelPredictions)
        print('RMSE:', test_results)
    
        # Set the column names to match the external results dataframe that we will join with later:
        columns = ['Regressor', 'Result']
        
        # Format results and return
        rmse_str = [str(test_results)] #make this a string and convert to a list
        Mtype = [Mtype] #make this a string
        result = spark.createDataFrame(zip(Mtype,rmse_str), schema=columns)
        result = result.withColumn('Result',result.Result.substr(0, 5))
        return result

    else:

        # Add parameters of your choice here:
        if Mtype in("RandomForestRegressor"):
            paramGrid = (ParamGridBuilder() \
#                            .addGrid(regressor.maxDepth, [2, 5, 10])
#                            .addGrid(regressor.maxBins, [5, 10, 20])
                           .addGrid(regressor.numTrees, [5, 20])
                         .build())

        # Add parameters of your choice here:
        if Mtype in("GBTRegressor"):
            paramGrid = (ParamGridBuilder() \
#                          .addGrid(regressor.maxDepth, [2, 5, 10, 20, 30]) \
                         .addGrid(regressor.maxBins, [10, 20]) \
                         .addGrid(regressor.maxIter, [10, 15])
                         .build())

        # Add parameters of your choice here:
        if Mtype in("DecisionTreeRegressor"):
            paramGrid = (ParamGridBuilder() \
#                          .addGrid(regressor.maxDepth, [2, 5, 10, 20, 30]) \
                         .addGrid(regressor.maxBins, [10, 20, 40]) \
                         .build())

        #Cross Validator requires all of the following parameters:
        crossval = CrossValidator(estimator=regressor,
                                  estimatorParamMaps=paramGrid,
                                  evaluator=RegressionEvaluator(metricName="rmse"),
                                  numFolds=5) # 3 is best practice
        # Fit Model: Run cross-validation, and choose the best set of parameters.
        fitModel = crossval.fit(train)
        
        # Get Best Model
        BestModel = fitModel.bestModel

        # FEATURE IMPORTANCES
        # Estimate of the importance of each feature.
        # Each feature’s importance is the average of its importance across all trees 
        # in the ensemble The importance vector is normalized to sum to 1. 
        print(" ")
        print('\033[1m' + Mtype," Feature Importances"+ '\033[0m')
        print("(Scores add up to 1)")
        print("Lowest score is the least important")
        print(" ")
        featureImportances = BestModel.featureImportances.toArray()
        # Convert from numpy array to list
        imp_scores = []
        for x in featureImportances:
            imp_scores.append(float(x))
        # Then zip with input_columns list and create a df
        result = spark.createDataFrame(zip(input_columns,imp_scores), schema=['feature','score'])
        print(result.orderBy(result["score"].desc()).show(truncate=False))
        
        #Create Global Variables for feature importances and models
        if Mtype in("DecisionTreeRegressor"):
            global DT_featureImportances
            DT_featureImportances = BestModel.featureImportances.toArray()
            global DT_BestModel 
            DT_BestModel = fitModel.bestModel
        if Mtype in("GBTRegressor"):
            global GBT_featureImportances
            GBT_featureImportances = BestModel.featureImportances.toArray()
            global GBT_BestModel 
            GBT_BestModel = fitModel.bestModel
        if Mtype in("RandomForestRegressor"):
            global RF_featureImportances
            RF_featureImportances = BestModel.featureImportances.toArray()
            global RF_BestModel 
            RF_BestModel = fitModel.bestModel
                    
        # Set the column names to match the external results dataframe that we will join with later:
        columns = ['Regressor', 'Result']
        
        # Make predictions.
        predictions = fitModel.transform(test)
        # Select (prediction, true label) and compute test error
        evaluator = RegressionEvaluator(metricName="rmse")
        rmse = evaluator.evaluate(predictions)
        rmse_str = [str(rmse)] #make this a string and convert to a list
        Mtype = [Mtype] #make this a string
        result = spark.createDataFrame(zip(Mtype,rmse_str), schema=columns)
        # Clean up the Result column and output
        result = result.withColumn('Result',result.Result.substr(0, 5))
        return result

In [14]:
regressors = [
                LinearRegression()
                ,DecisionTreeRegressor()
                ,RandomForestRegressor()
                ,GBTRegressor()
                ] 
    
columns = ['Regressor', 'Result']
vals = [("Place Holder","N/A")]
results = spark.createDataFrame(vals, columns)

for regressor in regressors:
    new_result = RegressTrainEval(regressor)
    results = results.union(new_result)
    
results = results.where("Regressor!='Place Holder'")
results.show(100,False)

Linear Regression Model Training Summary without cross validation:
 
Intercept: -93.90132483552297

Coefficients: 


+----------------+--------------------+
|feature         |coeff               |
+----------------+--------------------+
|age             |9.23797751383274    |
|superplasticizer|0.1595699388147034  |
|cement          |0.13896160826478404 |
|slag            |0.1197160979091706  |
|flyash          |0.0980653705217326  |
|fineaggregate   |0.03854192758838611 |
|coarseaggregate |0.035840885549187955|
|water           |-0.10595204384422699|
+----------------+--------------------+

None
numIterations: 0
objectiveHistory: (scaled loss + regularization) at each iteration 
 [0.0]

Training RMSE: 7.010572
Training r2: 0.825331

Test RMSE: 7.645285919583659
Test r2: 0.7847055301005809

Linear Regression Model Summary WITH cross validation:
 


Coefficients: 


+----------------+--------------------+
|feature         |coeff               |
+----------------+--------------------+
|age             |9.228829767853272   |
|superplasticizer|0.15959505683051387 |
|cement          |0.13757784913430063 |
|slag            |0.11810528026458067 |
|flyash          |0.09613745948626648 |
|fineaggregate   |0.03696349050173879 |
|coarseaggregate |0.034553166505893344|
|water           |-0.11069765460500015|
+----------------+--------------------+

None
Coefficient Standard Errors: 


+----------------+--------------------+
|feature         |coeff std error     |
+----------------+--------------------+
|age             |0.23325943558234039 |
|superplasticizer|0.07327900588760776 |
|water           |0.03200368775203396 |
|flyash          |0.010033738212670382|
|fineaggregate   |0.008497578575414421|
|slag            |0.008126917876082499|
|coarseaggregate |0.007568443764794181|
|cement          |0.006874651821113896|
+----------------+--------------------+

None
 
P Values: 


+----------------+--------------------+
|feature         |P-Value             |
+----------------+--------------------+
|superplasticizer|0.029722402919157842|
|water           |5.728815899197492E-4|
|fineaggregate   |1.549771806641509E-5|
|coarseaggregate |5.820802545741799E-6|
|flyash          |0.0                 |
|age             |0.0                 |
|slag            |0.0                 |
|cement          |0.0                 |
+----------------+--------------------+

None
 
RMSE: 7.640985048251783


... (3 outputs left out)

+----------------+--------------------+
|feature         |score               |
+----------------+--------------------+
|age             |0.3636408650479469  |
|cement          |0.26642036733368074 |
|water           |0.11328073664686111 |
|superplasticizer|0.09873874569703629 |
|slag            |0.04919165349877637 |
|flyash          |0.04893651011838101 |
|fineaggregate   |0.03870423009269698 |
|coarseaggregate |0.021086891564620618|
+----------------+--------------------+

None


 
GBTRegressor  Feature Importances
(Scores add up to 1)
Lowest score is the least important
 


+----------------+-------------------+
|feature         |score              |
+----------------+-------------------+
|age             |0.27880148934904403|
|cement          |0.2443229987561632 |
|water           |0.11347478088208036|
|coarseaggregate |0.0961646419601923 |
|slag            |0.09398857998392418|
|fineaggregate   |0.06623780965854144|
|superplasticizer|0.06582270262396395|
|flyash          |0.04118699678609047|
+----------------+-------------------+

None


+---------------------+------+
|Regressor            |Result|
+---------------------+------+
|LinearRegression     |7.640 |
|DecisionTreeRegressor|8.450 |
|RandomForestRegressor|7.786 |
|GBTRegressor         |6.769 |
+---------------------+------+



### **Which features are the strongest predictors of cement strength?**

> The Gradient Boosted regressor had the lowest root mean squared error (rmse) which would make it the best performing model.

> Looks like age, cement and water were the most influencial features in the dataset which we can see by looking at the coefficients and feature important scores in the GBT model.

## 2. For the following given inputs, what would be the estimated cement strength?

- Cement: 540
- Blast Furnace Slag: 0
- Fly Ash: 0
- Water: 162
- Superplasticizer: 2.5
- Coarse Aggregate: 1040
- Fine Aggregate: 676
- Age: 28

The correct answer is 79.99. Let's how close your prediction is!

In [15]:
from pyspark.sql.types import FloatType

input_ = [(540,
          0, 
          0, 
          162, 
          2.5, 
          1040, 
          676, 
          28)]

data = spark.createDataFrame(input_, schema=input_columns)

In [16]:
data.show(truncate=False)

+------+----+------+-----+----------------+---------------+-------------+---+
|cement|slag|flyash|water|superplasticizer|coarseaggregate|fineaggregate|age|
+------+----+------+-----+----------------+---------------+-------------+---+
|540   |0   |0     |162  |2.5             |1040           |676          |28 |
+------+----+------+-----+----------------+---------------+-------------+---+



In [17]:
# The model was trained on a transformed age: capped to the 1st-99th percentile, then log(age + 1).
# New inputs have to go through the same transform, or the model sees ages far outside what it was trained on.
age_low, age_high = concrete_data.approxQuantile('age', [0.01, 0.99], 0.25)

def prepare_input(df):
    capped_age = when(col('age') < age_low, age_low).when(col('age') > age_high, age_high).otherwise(col('age'))
    df = df.withColumn('age', log(capped_age + 1))
    return VectorAssembler(inputCols=input_columns, outputCol='features').transform(df)

input_data = prepare_input(data).select('features')

In [18]:
input_data.show(truncate=False)

+--------------------------------------------------------+
|features                                                |
+--------------------------------------------------------+
|[540.0,0.0,0.0,162.0,2.5,1040.0,676.0,3.367295829986474]|
+--------------------------------------------------------+



In [19]:
GBT_BestModel.transform(input_data).withColumnRenamed('prediction', 'csMPa') \
                .withColumn('csMPa', round(col('csMPa'), 2)).show(truncate=False)

+--------------------------------------------------------+-----+
|features                                                |csMPa|
+--------------------------------------------------------+-----+
|[540.0,0.0,0.0,162.0,2.5,1040.0,676.0,3.367295829986474]|66.99|
+--------------------------------------------------------+-----+



## 3. Now see if you can ask users to input their own value for Age and return a predicted value for the cement stength. 

We did not cover this is in the lecture so you'll have to put your thinking cap on. Accepting user input in PySpark works just like it does in traditional Python.
<br>

val = input("Enter your value: ") 

In [20]:
try:
    input_age = int(input("Enter your value for age (days): "))
except Exception:  # no interactive prompt, e.g. when the notebook runs headless
    input_age = 28
    print('No input available, using age =', input_age)

data = data.withColumn('age', lit(input_age))
GBT_BestModel.transform(prepare_input(data).select('features')).withColumnRenamed('prediction', 'csMPa') \
                .withColumn('csMPa', round(col('csMPa'), 2)).show(truncate=False)

No input available, using age = 28


+--------------------------------------------------------+-----+
|features                                                |csMPa|
+--------------------------------------------------------+-----+
|[540.0,0.0,0.0,162.0,2.5,1040.0,676.0,3.367295829986474]|66.99|
+--------------------------------------------------------+-----+



## 4. Make recommendations of optimal values for cement ingredients (our features)

See if you can find the optimal amount of cement to recommend holding the rest of the values from the previous question constant, assuming that the higher the cement strength value the better. 

In [21]:
concrete_data.select('cement').summary('min', 'max').show(truncate=False)

+-------+------+
|summary|cement|
+-------+------+
|min    |102.0 |
|max    |540.0 |
+-------+------+



In [22]:
# Score every cement amount from the minimum to the maximum in the data, in steps of 5,
# holding the other ingredients at the values above, and pick the highest predicted strength.
# (Stopping at the first plateau doesn't work here: gradient-boosted trees predict in flat steps.)
base = data.first().asDict()
cement_values = list(range(100, 545, 5))
grid = spark.createDataFrame(
    [tuple(float(c) if name == 'cement' else float(base[name]) for name in input_columns) for c in cement_values],
    schema=input_columns)
curve = GBT_BestModel.transform(prepare_input(grid)) \
                .select(grid.columns[0], round(col('prediction'), 2).alias('csMPa')).toPandas()
best = curve.loc[curve['csMPa'].idxmax()]
print('The optimal amount of cement is {:.0f} kg/m3, with a predicted strength of {} MPa.'.format(best['cement'], best['csMPa']))

The optimal amount of cement is 390 kg/m3, with a predicted strength of 66.99 MPa.


In [23]:
# How the predicted strength changes with cement (every 50 kg/m3)
curve[curve['cement'] % 50 == 0].reset_index(drop=True)

,cement,csMPa
0,100.0,28.48
1,150.0,27.20
2,200.0,41.33
3,250.0,43.93
4,300.0,44.75
5,350.0,46.72
6,400.0,66.99
7,450.0,66.99
8,500.0,66.99
